In [3]:
import codecs
import sys
import multiprocessing
import json
import pandas as pd
import jsonlines
import os
from vllm import LLM, SamplingParams
import torch

path = "Qwen3-0.6B"

def inf(input_path, output_json_path, start_index, end_index):
    prompts, output_list, results, ids = [], [], [], []
    with codecs.open(input_path, "r",encoding='utf-8') as fin:
        line_index = 0
        input_list = []
        data_list = []
        for line in fin:
            line_index += 1
            line = line.strip()
            if not line:
                continue
            try:
                data = json.loads(line)
                input_ = data.get('input', '') if data.get('input', None) else data.get('data', '')
                if len(input_)>8000:continue
                assert input_!=''
                input_list.append(input_)
                data_list.append(data)
            except Exception as e:
                print(line)
        prompts = ["<|im_start|>system\n<|im_end|>\n<|im_start|>user\n" + input + "\n<|im_end|>\n<|im_start|>assistant\n"
                      for input in input_list][int(start_index): int(end_index)]
    llm = LLM(model=path, trust_remote_code=True, tensor_parallel_size=1, gpu_memory_utilization=0.8)

    # sampling_params = SamplingParams(temperature=0.0, max_tokens=4096)
    sampling_params = SamplingParams(repetition_penalty=1.05, temperature= 0.6, top_p=0.95, top_k=20, max_tokens=8096)
    print('Start generation --------------------')
    outputs = llm.generate(prompts, sampling_params)

    results = [i.outputs[0].text for i in outputs]
    qwen_input_list = [i.prompt for i in outputs]
    print('prompts', len(prompts), 'results', len(results), 'output_list', len(output_list))
    # Print the outputs.

    # save as jsonl
    save_path = os.path.join(output_json_path, os.path.basename(input_path).split('.')[0] + '-' + str(start_index) + '-' + str(end_index) + '.jsonl')
    print(save_path)
    with jsonlines.open(save_path, mode='w') as writer:
        for i in range(len(outputs)):
            data = data_list[i+int(start_index)]
            data["qwen_input"] = qwen_input_list[i]
            data["qwen_output"] = results[i]
            writer.write(data)
    print('finish generation & saving')



In [2]:
!pip install jsonlines

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [4]:
input_path = "opencompass_code_debug_100.jsonl"
output_json_path= "./"
start_index=0 
end_index=10
inf(input_path, output_json_path, start_index, end_index)

INFO 06-30 23:19:17 config.py:549] This model supports multiple tasks: {'score', 'classify', 'embed', 'reward', 'generate'}. Defaulting to 'generate'.
WARNING 06-30 23:19:17 arg_utils.py:1187] Chunked prefill is enabled by default for models with max_model_len > 32K. Currently, chunked prefill might not work with some features or models. If you encounter any issues, please disable chunked prefill by setting --enable-chunked-prefill=False.
INFO 06-30 23:19:17 config.py:1555] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 06-30 23:19:17 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.3) with config: model='Qwen3-0.6B', speculative_config=None, tokenizer='Qwen3-0.6B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_re

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 06-30 23:19:19 model_runner.py:1115] Loading model weights took 1.1103 GB
INFO 06-30 23:19:19 worker.py:267] Memory profiling takes 0.34 seconds
INFO 06-30 23:19:19 worker.py:267] the current vLLM instance can use total_gpu_memory (23.57GiB) x gpu_memory_utilization (0.80) = 18.86GiB
INFO 06-30 23:19:19 worker.py:267] model weights take 1.11GiB; non_torch_memory takes 0.00GiB; PyTorch activation peak memory takes 1.38GiB; the rest of the memory reserved for KV Cache is 16.36GiB.
INFO 06-30 23:19:20 executor_base.py:111] # cuda blocks: 9575, # CPU blocks: 2340
INFO 06-30 23:19:20 executor_base.py:116] Maximum concurrency for 40960 tokens per request: 3.74x


OutOfMemoryError: CUDA out of memory. Tried to allocate 600.00 MiB. GPU 0 has a total capacity of 23.57 GiB of which 534.81 MiB is free. Including non-PyTorch memory, this process has 23.03 GiB memory in use. Of the allocated memory 22.07 GiB is allocated by PyTorch, with 28.00 MiB allocated in private pools (e.g., CUDA Graphs), and 112.92 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)